# CASIA v2 Forgery Detection -- End-to-End Pipeline

**Runtime:** 60-70 min on Kaggle T4 x2 (enable GPU accelerator before running)  
**Dataset:** `namtpham/casia2groundtruth` (~7k authentic + ~5k tampered images)  
**Run all cells in order.** Do not skip Section A.

| Section | Cells | Content |
|---------|-------|---------|
| A: Data Preparation | 2-6 | Download, EDA, stratified 80/10/10 split |
| B: Model Training | 8-16 | 4 configs, training loop, Grad-CAM, metrics |
| C: Summary | 18 | Output file listing + next steps |

In [ ]:
# ---- pip install (torch is pre-installed on Kaggle; skip to avoid CUDA conflicts) ----
import subprocess

for pkg in [
    'kagglehub',
    'timm==1.0.11',
    'albumentations==1.4.21',
    'opencv-python-headless==4.10.0.84',
    'grad-cam==1.5.4',
    'scikit-learn==1.5.2',
]:
    subprocess.run(['pip', 'install', pkg, '--quiet'], check=True)

# ---- seeds (set once for the entire notebook) ----
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- imports ----
import io
import os
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from dataclasses import dataclass
from pathlib import Path
from PIL import Image
from typing import Callable

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
)

# ---- global constants ----
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SPLITS_DIR = '/kaggle/working/splits'
EPOCHS     = 3
BATCH_SIZE = 32
LR         = 1e-4

print(f'torch={torch.__version__} | cuda={torch.cuda.is_available()} | device={DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Setup complete.')

In [ ]:
import kagglehub

DATASET_NAME = 'namtpham/casia2groundtruth'
dataset_path = kagglehub.dataset_download(DATASET_NAME)
print(f'Dataset root: {dataset_path}')

# Walk directory tree; infer label from folder name
# CASIA v2 layout: Au/ = authentic (0), Tp/ = tampered (1)
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

records = []
for root, _, files in os.walk(dataset_path):
    for fname in files:
        if Path(fname).suffix.lower() not in IMAGE_EXTS:
            continue
        fpath = os.path.join(root, fname)
        label = None
        for part in Path(fpath).parts:
            pl = part.lower()
            if pl == 'au' or pl.startswith('au_') or pl.startswith('au '):
                label = 0
                break
            elif pl == 'tp' or pl.startswith('tp_') or pl.startswith('tp '):
                label = 1
                break
        if label is not None:
            records.append({'path': fpath, 'label': label})

# df['path'] is absolute -- valid for all cells in this notebook
df = pd.DataFrame(records)
print(f'Total images: {len(df)}')
print(df['label'].value_counts().rename({0: 'authentic', 1: 'tampered'}).to_string())

In [ ]:
# EDA: class distribution
label_names  = {0: 'authentic', 1: 'tampered'}
label_counts = df['label'].value_counts().sort_index()
labels_str   = [label_names[k] for k in label_counts.index]
colors       = ['steelblue', 'tomato']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

bars = axes[0].bar(labels_str, label_counts.values, color=colors, width=0.5)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Image count')
for bar, v in zip(bars, label_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, v + 50,
                 str(v), ha='center', fontweight='bold')

axes[1].pie(label_counts.values, labels=labels_str, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Class Ratio')

plt.suptitle('CASIA v2 -- Class Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Authentic: {label_counts[0]}  |  Tampered: {label_counts[1]}')

In [ ]:
# Display 3 samples per class
fig, axes = plt.subplots(2, 3, figsize=(13, 9))

for row_idx, (label_val, label_name) in enumerate(label_names.items()):
    samples = df[df['label'] == label_val].sample(3, random_state=SEED)
    for col, (_, record) in enumerate(samples.iterrows()):
        ax = axes[row_idx][col]
        try:
            img = mpimg.imread(record['path'])
        except Exception as e:
            ax.set_title(f'Error: {e}', fontsize=7)
            ax.axis('off')
            continue
        fname = Path(record['path']).name
        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.set_title('[' + label_name + ']\n' + fname, fontsize=8)
        ax.axis('off')

plt.suptitle('3 Samples per Class  (authentic = row 0, tampered = row 1)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/samples_per_class.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Stratified 80 / 10 / 10 split
os.makedirs(SPLITS_DIR, exist_ok=True)

train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED
)

for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    out = split_df.copy()
    out['dataset_split'] = split_name
    csv_path = f'{SPLITS_DIR}/{split_name}.csv'
    out[['path', 'label', 'dataset_split']].to_csv(csv_path, index=False)
    n_auth = (out['label'] == 0).sum()
    n_tamp = (out['label'] == 1).sum()
    print(f'{split_name:5s}: {len(out):5d} total | {n_auth:4d} authentic | {n_tamp:4d} tampered')

print()
print('Paths in CSV are absolute -- valid for all subsequent cells in this notebook.')

## Section B: Model Training & Comparison

Four configurations trained on the same splits (3 epochs each, ~15 min/model on T4):

| ID | Model | Strategy | Input |
|----|-------|----------|-------|
| A  | ResNet50 | Full fine-tune | RGB |
| B  | EfficientNet-B0 | Full fine-tune | RGB |
| C  | EfficientNet-B0 | Head-only (frozen backbone) | RGB |
| D  | ResNet50 | Full fine-tune | ELA (JPEG q=75) |

Training config: AdamW lr=1e-4, weight_decay=1e-4, CosineAnnealingLR, batch=32, AMP fp16.

In [ ]:
def compute_ela(image_path: str, quality: int = 75) -> Image.Image:
    """
    Error Level Analysis:
    1. Re-save image as JPEG at given quality into a BytesIO buffer.
    2. Compute absolute diff between original and re-compressed.
    3. Scale x10, clip [0, 255].
    Returns: RGB PIL Image (same spatial size as input).
    """
    orig    = Image.open(image_path).convert('RGB')
    orig_np = np.array(orig, dtype=np.float32)

    buf = io.BytesIO()
    orig.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    comp_np = np.array(Image.open(buf).convert('RGB'), dtype=np.float32)

    diff = np.clip(np.abs(orig_np - comp_np) * 10.0, 0, 255).astype(np.uint8)
    return Image.fromarray(diff)

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# Only horizontal flip + color jitter -- preserve forgery boundary artifacts
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.0, p=0.5),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

print('Transforms defined.')

In [ ]:
class ForgedDocDataset(Dataset):
    def __init__(self, csv_path: str, transform=None, use_ela: bool = False):
        self.df        = pd.read_csv(csv_path).reset_index(drop=True)
        self.transform = transform
        self.use_ela   = use_ela

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        # path is absolute (written by Cell 6 using kagglehub download root)
        if self.use_ela:
            img = compute_ela(row['path'])
        else:
            img = Image.open(row['path']).convert('RGB')
        img_np = np.array(img)
        if self.transform:
            img_np = self.transform(image=img_np)['image']
        return img_np, int(row['label'])


def make_loaders(use_ela: bool = False, batch_size: int = 32):
    kw = dict(num_workers=2, pin_memory=True)
    return (
        DataLoader(ForgedDocDataset(f'{SPLITS_DIR}/train.csv', train_transform, use_ela),
                   batch_size=batch_size, shuffle=True,  **kw),
        DataLoader(ForgedDocDataset(f'{SPLITS_DIR}/val.csv',   val_transform,   use_ela),
                   batch_size=batch_size, shuffle=False, **kw),
        DataLoader(ForgedDocDataset(f'{SPLITS_DIR}/test.csv',  val_transform,   use_ela),
                   batch_size=batch_size, shuffle=False, **kw),
    )

print('ForgedDocDataset + make_loaders defined.')

In [ ]:
@dataclass
class ModelConfig:
    name:     str
    build_fn: Callable
    use_ela:  bool = False


def _trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_resnet50() -> nn.Module:
    m = timm.create_model('resnet50', pretrained=True, num_classes=2)
    print(f'  resnet50 full-tune       : {_trainable(m):>12,} trainable params')
    return m


def build_efficientnet_full() -> nn.Module:
    m = timm.create_model('tf_efficientnet_b0', pretrained=True, num_classes=2)
    print(f'  efficientnet_b0 full     : {_trainable(m):>12,} trainable params')
    return m


def build_efficientnet_head() -> nn.Module:
    m = timm.create_model('tf_efficientnet_b0', pretrained=True, num_classes=2)
    for name, p in m.named_parameters():
        p.requires_grad = 'classifier' in name or 'fc' in name
    n = _trainable(m)
    print(f'  efficientnet_b0 head-only: {n:>12,} trainable params')
    assert n > 0, 'No trainable params -- check classifier layer name in timm model'
    return m


def build_resnet50_ela() -> nn.Module:
    m = timm.create_model('resnet50', pretrained=True, num_classes=2)
    print(f'  resnet50+ELA full-tune   : {_trainable(m):>12,} trainable params')
    return m


MODEL_CONFIGS = [
    ModelConfig('ResNet50_full',        build_resnet50,          use_ela=False),
    ModelConfig('EfficientNet_B0_full', build_efficientnet_full, use_ela=False),
    ModelConfig('EfficientNet_B0_head', build_efficientnet_head, use_ela=False),
    ModelConfig('ResNet50_ELA',         build_resnet50_ela,      use_ela=True),
]

print('\nModel configs registered:')
for cfg in MODEL_CONFIGS:
    print(f'  {cfg.name:<30s} | use_ela={cfg.use_ela}')

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss = criterion(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    total_loss = 0.0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss   = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)
            total_loss += loss.item() * imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return {
        'loss':      total_loss / len(loader.dataset),
        'accuracy':  accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, zero_division=0),
        'recall':    recall_score(all_labels, all_preds, zero_division=0),
        'f1':        f1_score(all_labels, all_preds, zero_division=0),
        'auc':       roc_auc_score(all_labels, all_probs),
    }


def measure_latency(model: nn.Module, n_samples: int = 100, img_size: int = 224) -> float:
    """Mean CPU inference latency (ms/image). Warmup=5 iters, forward-pass only."""
    cpu_model = model.cpu().eval()
    dummy = torch.randn(1, 3, img_size, img_size)
    with torch.no_grad():
        for _ in range(5):       # warmup
            cpu_model(dummy)
        times = []
        for _ in range(n_samples):
            t0 = time.perf_counter()
            cpu_model(dummy)
            times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times))


print('Training utilities defined.')

In [ ]:
results = {}
sep = '=' * 64

for cfg in MODEL_CONFIGS:
    print()
    print(sep)
    print(f' Training: {cfg.name}')
    print(sep)

    train_loader, val_loader, test_loader = make_loaders(cfg.use_ela, BATCH_SIZE)
    model = cfg.build_fn().to(DEVICE)

    trainable_params = _trainable(model)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()
    scaler    = torch.cuda.amp.GradScaler()

    train_losses, val_losses = [], []

    for epoch in range(1, EPOCHS + 1):
        t_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, DEVICE)
        v_met  = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()
        train_losses.append(t_loss)
        val_losses.append(v_met['loss'])
        vl, va, vf = v_met['loss'], v_met['accuracy'], v_met['f1']
        print(f'  Epoch {epoch}/{EPOCHS} | train={t_loss:.4f} | val_loss={vl:.4f} | val_acc={va:.4f} | val_f1={vf:.4f}')

    test_met = evaluate(model, test_loader, criterion, DEVICE)
    latency  = measure_latency(model)    # temporarily moves model to CPU
    model.to(DEVICE)                     # restore to GPU for Grad-CAM

    results[cfg.name] = {
        **{f'test_{k}': v for k, v in test_met.items()},
        'latency_ms':       latency,
        'trainable_params': trainable_params,
        'train_losses':     train_losses,
        'val_losses':       val_losses,
        'model':            model,
        'use_ela':          cfg.use_ela,
    }

    ta, tp, tr, tf1, tauc = (
        test_met['accuracy'], test_met['precision'], test_met['recall'],
        test_met['f1'], test_met['auc'],
    )
    print(f'  TEST  | acc={ta:.4f} | prec={tp:.4f} | rec={tr:.4f} | f1={tf1:.4f} | auc={tauc:.4f} | latency={latency:.2f}ms')

print()
print('All models trained.')

In [ ]:
# Save metrics.csv
rows = []
for name, res in results.items():
    rows.append({
        'model':            name,
        'accuracy':         round(res['test_accuracy'],  4),
        'precision':        round(res['test_precision'], 4),
        'recall':           round(res['test_recall'],    4),
        'f1':               round(res['test_f1'],        4),
        'auc':              round(res['test_auc'],       4),
        'latency_ms':       round(res['latency_ms'],     2),
        'trainable_params': res['trainable_params'],
    })

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv('/kaggle/working/metrics.csv', index=False)
print(metrics_df.to_string(index=False))
print()
print('Saved: /kaggle/working/metrics.csv')

In [ ]:
# Training curves -- 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, res) in enumerate(results.items()):
    ax = axes[idx]
    ep = range(1, EPOCHS + 1)
    ax.plot(ep, res['train_losses'], 'b-o', label='train loss', linewidth=2, markersize=6)
    ax.plot(ep, res['val_losses'],   'r-o', label='val loss',   linewidth=2, markersize=6)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (CrossEntropy)')
    ax.set_xticks(list(ep))
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Training Curves -- 4 Model Configurations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /kaggle/working/training_curves.png')

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

os.makedirs('/kaggle/working/gradcam', exist_ok=True)

_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def get_target_layer(model: nn.Module, model_name: str):
    """ResNet50 -> layer4[-1]; EfficientNet-B0 -> conv_head (clearer than blocks[-1][-1])."""
    if 'ResNet' in model_name:
        return [model.layer4[-1]]
    return [model.conv_head]


def collect_tp_samples(model, loader, device, n=3):
    """Collect up to n (input_tensor, display_rgb_float32) True Positive pairs."""
    model.eval()
    tensors, imgs_np = [], []
    with torch.no_grad():
        for batch_imgs, batch_labels in loader:
            preds = model(batch_imgs.to(device)).argmax(dim=1).cpu()
            for img_t, label, pred in zip(batch_imgs, batch_labels, preds):
                if int(label) == 1 and int(pred) == 1:   # True Positive (tampered)
                    disp = (img_t.permute(1, 2, 0).numpy() * _STD + _MEAN)
                    disp = disp.clip(0, 1).astype(np.float32)
                    tensors.append(img_t.unsqueeze(0))
                    imgs_np.append(disp)
                if len(tensors) >= n:
                    return tensors, imgs_np
    return tensors, imgs_np


def save_gradcam(model, model_name, use_ela, n_tp=3):
    _, _, test_loader = make_loaders(use_ela=use_ela, batch_size=8)
    tensors, imgs_np  = collect_tp_samples(model, test_loader, DEVICE, n=n_tp)

    if not tensors:
        print(f'  WARNING: no TP samples found for {model_name}')
        return

    cam   = GradCAM(model=model, target_layers=get_target_layer(model, model_name))
    ncols = len(tensors)
    fig, axes = plt.subplots(2, ncols, figsize=(5 * ncols, 10))
    if ncols == 1:
        axes = np.expand_dims(axes, axis=1)

    for col, (tensor, img_np) in enumerate(zip(tensors, imgs_np)):
        grayscale = cam(
            input_tensor=tensor.to(DEVICE),
            targets=[ClassifierOutputTarget(1)],
        )[0]
        overlay = show_cam_on_image(img_np, grayscale, use_rgb=True)

        axes[0][col].imshow(img_np)
        axes[0][col].set_title(f'Original TP #{col + 1}', fontsize=9)
        axes[0][col].axis('off')
        axes[1][col].imshow(overlay)
        axes[1][col].set_title(f'Grad-CAM TP #{col + 1}', fontsize=9)
        axes[1][col].axis('off')

    plt.suptitle(f'Grad-CAM -- {model_name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    out_path = f'/kaggle/working/gradcam/{model_name}_gradcam.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {out_path}')


for cfg in MODEL_CONFIGS:
    print(f'\nGrad-CAM: {cfg.name}')
    save_gradcam(results[cfg.name]['model'], cfg.name, cfg.use_ela)

print()
print('Grad-CAM complete.')

## Results & Discussion

### Metrics Table

*After the run, copy values from `/kaggle/working/metrics.csv` into this table:*

| Model | Acc | Precision | Recall | F1 | AUC | Latency (ms/img) | Trainable Params |
|-------|-----|-----------|--------|----|-----|-----------------|------------------|
| ResNet50_full | | | | | | | ~23.5 M |
| EfficientNet_B0_full | | | | | | | ~5.3 M |
| EfficientNet_B0_head | | | | | | | ~1.3 K |
| ResNet50_ELA | | | | | | | ~23.5 M |

---

### Architecture: ResNet50 vs EfficientNet-B0 (A vs B)

Both configs use full fine-tune on RGB, identical hyperparameters. EfficientNet-B0 has ~4x fewer
parameters but applies compound scaling (depth + width + resolution), which often matches or
exceeds ResNet50 accuracy. For forgery detection -- a texture-heavy, low-level signal task --
EfficientNet-B0's multi-scale features can capture subtle compression artifacts more efficiently.
If B achieves similar F1 with lower latency, it is the preferred production architecture.

### Full Fine-tune vs Head-only (B vs C)

Config C freezes the backbone (~5.3 M params frozen) and trains only the linear classifier
(~1.3 K params). Expected: faster convergence but a lower accuracy ceiling, because ImageNet
features were not optimized for detecting JPEG compression artifacts. Head-only is justified when:
(a) labelled data is very scarce (<1 k images) and overfitting risk is high, or
(b) compute budget is extremely tight. With ~9 k training images, full fine-tune (Config B)
should clearly outperform head-only.

### ELA Preprocessing: ResNet50 RGB vs ResNet50+ELA (A vs D)

ELA highlights regions re-saved at a different JPEG quality than the rest of the image --
a direct signal for spliced or copy-moved regions. Config D feeds the ELA map (RGB 3-channel)
into the same ResNet50 without modifying the architecture.

**When ELA helps (D > A):**
- Tampered region is a JPEG splice pasted into a PNG original (double-compression mismatch).
- Forgery is localized and the compression grid is misaligned with the host image.

**When ELA does NOT help (A >= D):**
- Original was already JPEG-compressed before tampering (ELA noise drowns the forgery signal).
- Forgery was saved at the same quality level (no ELA contrast between tampered/authentic regions).
- Images are lossless PNG with no compression artifacts (ELA output near-zero everywhere).

**Production recommendation:**
- General-purpose detector: EfficientNet-B0 full fine-tune (B) -- best accuracy/latency tradeoff.
- JPEG-splicing specialist: ensemble A (RGB) + D (ELA) -- ELA adds signal RGB misses.
- Head-only (C): only if labelled data < 1 k or compute is severely constrained.

### Grad-CAM Observations

- Configs A/B (RGB): attention spreads over texture boundaries and high-frequency edges.
- Config D (ELA): attention concentrates on the spliced region where the ELA diff is highest;
  more interpretable for forensic analysts.
- Config C (head-only): heatmaps may be diffuse; backbone was not tuned for this task.

## Summary of Outputs

All files saved to `/kaggle/working/`:

| File | Description |
|------|-------------|
| `class_distribution.png` | EDA bar chart + pie chart |
| `samples_per_class.png` | 3 sample images per class |
| `splits/train.csv` | 80 % training split (~9,600 images) |
| `splits/val.csv` | 10 % validation split (~1,200 images) |
| `splits/test.csv` | 10 % test split (~1,200 images) |
| `metrics.csv` | Acc / F1 / AUC / latency for 4 models |
| `training_curves.png` | Loss curves, 2x2 subplots |
| `gradcam/ResNet50_full_gradcam.png` | 3 TP Grad-CAM samples |
| `gradcam/EfficientNet_B0_full_gradcam.png` | 3 TP Grad-CAM samples |
| `gradcam/EfficientNet_B0_head_gradcam.png` | 3 TP Grad-CAM samples |
| `gradcam/ResNet50_ELA_gradcam.png` | 3 TP Grad-CAM samples |

**Next steps:**
1. Download `metrics.csv` -> paste values into the table in Section B above.
2. Download `gradcam/` folder for portfolio screenshots.
3. Copy `training_curves.png` and `gradcam/` to `06-forged-doc/results/` in the repo.